In [ ]:
import pandas as pd
import sagemaker
import torch
import numpy as np
from torch.utils.data import DataLoader,Dataset
from transformers import DistilBertTokenizer,DistilBertModel
from tqdm import tqdm
import argparse
import pandas as pd


import pandas as pd
s3_path='s3://multi-class-text-classification-data/training/newsCorpora.csv'
df=pd.read_csv(s3_path,sep="\t",names=['ID','TITLE','URL','PUBLISHER','CATEGORY','STORY','HOSTNAME','TIMESTAMP'])

df_work=df.copy()
df_work=df_work[['TITLE','CATEGORY']]

my_dict={
    'e':'Entertainment',
    'b':'Business',
    't':'Science',
    'm':'Health'
}

def update_category(x):
    return my_dict[x]


df_work['CATEGORY']=df_work['CATEGORY'].apply(lambda x: update_category(x))

print(df)


f=df.sample(frac=0.25,random_state=1)
f=f.reset_index(drop=True)

tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class NewsDataset(Dataset):
    def __init__(self,dataframe,tokenizer,max_length):
        self.len=len(dataframe)
        self.data= Dataframe
        self.tokenizer=tokenizer
        self.max_length=max_length

    def __getitem__(self,index):
        title=str(self.data.TITLE[index])
        title=" ".join(title.split())

        inputs= self.tokenizer.encode_plus(
                title,
                None,
                add_special_tokens= True,
                max_length=self.max_length,
                padding='max_length',
                return_token_type_ids=True,
                truncations= True,
                return_attention_mask= True
            )
        ids=inputs['input_ids']
        mask= inputs['attention_mask']

        return {
            'ids': torch.tensor(ids,dtype=torch.long),
            'mask': torch.tensor(mask,dtype=torch.long),
            'targets': torch.tensor(self.data.CATEGORY[index],dtype=torch.long)
        }
    def __len__(self):
        return self.len


training_size=0.8

train_dataset=df.sample(frac=trining_size,random_state=200)
test_dataset=df.drop(train_dataset.index).reset_index(drop=True)

train_dataset=train_dataset.reset_index(drop=True)

print(f"Full Dataset {df.shape}")
print(f"Trian Dataset {train_dataset.shape}")
print(f"Test Dataset {test_dataset.shape}")

MAX_LEN=512
TRAIN_BATCH_SIZE=4
VALID_BATCH_SIZE=2

training_set=NewsDatasets(train_dataset,tokenizer,MAX_LEN)
test_set=NewsDatasets(test_dataset,tokenizer,MAX_LEN)

train_parameters={
    'batch_size':TRAIN_BATCH_SIZE,
    'shuffle': True,
    'num_workers': 0
}

test_parameters={
    'batch_size':VALID_BATCH_SIZE,
    'shuffle': True,
    'num_workers': 0
}

training_loader=DataLoader(training_set,**train_parameters)
test_loader=DataLoader(training_set,**test_parameters)